In this program we are going to upload datasets to mysql as tables so that we can do querying and answer business questions there

Creating database

In [6]:
# Loading environment files
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv
import os

load_dotenv()

True

In [7]:
print(os.getenv('DB_USER'))

root


In [12]:
# Creating database
# Connect without specifying a database
url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASS"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT"))
)

engine = create_engine(url)

# Testing engine connectivity
with engine.connect() as conn:
    result = conn.execute(text("SHOW DATABASES"))
    for row in result:
        print(row[0])

ecommerce
information_schema
mysql
performance_schema
sakila
sys


In [11]:

with engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {os.getenv('DB_NAME')}"))
    conn.commit()

print(f"Database '{os.getenv('DB_NAME')}' created (if it didn't exist).")

Database 'ecommerce' created (if it didn't exist).


In [20]:
from pathlib import Path
import pandas as pd

# Connecting to a specific database
url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASS"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME")
)
engine = create_engine(url)

folder_path = os.getcwd()
datasets_folder_path = os.path.join(folder_path,"..\\data_model\\")
csv_folder = Path(datasets_folder_path)

for csv_file in csv_folder.glob("*.csv"):
    table_name = csv_file.stem.lower().replace(" ", "_")
    df = pd.read_csv(csv_file)

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",  # replace, append, or fail
        index=False
    )

    print(f"Uploaded {csv_file.name} -> {table_name}")



Uploaded dim_customers.csv -> dim_customers
Uploaded dim_products.csv -> dim_products
Uploaded dim_sellers.csv -> dim_sellers
Uploaded facts_order_items.csv -> facts_order_items
